# BigQuery Hybrid Search — Feature Walkthrough

End-to-end demo of every public feature of
[`langchain-bigquery-hybridsearch`](../../README.md).

**What you will see**

1. Initialize `BigQueryHybridSearchVectorStore` (table is auto-created).
2. Inherited `similarity_search` (sanity check).
3. **Hybrid search — pre-filter mode** (`SEARCH()` ➜ `VECTOR_SEARCH()`).
4. **Hybrid search — RRF mode** (Reciprocal Rank Fusion).
5. RRF tuning via `rrf_k`.
6. Metadata filtering (dict + raw SQL).
7. Analyzer / search-field configuration (`PATTERN_ANALYZER`).
8. Retriever interface — `as_retriever(search_type="hybrid")` + LCEL.
9. Cleanup.

**Prerequisites** — see [README.md → Integration Tests](../../README.md#integration-tests):

- `gcloud auth application-default login`
- BigQuery API + Vertex AI API enabled on your project
- IAM roles: `bigquery.dataEditor`, `bigquery.jobUser`, `aiplatform.user`
- Optional: `bq mk --dataset <project>:<dataset>` (or grant `bigquery.datasets.create`)

## 1. Setup

### Install extra dependencies

The package itself only requires `langchain-google-community[featurestore]`.
For this notebook we additionally need the Vertex AI embedding wrapper and
`python-dotenv` (already in the `dev` extra).

```bash
uv pip install langchain-google-vertexai python-dotenv
```

In [1]:
import os
import uuid
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from google.cloud import bigquery
from langchain_google_vertexai import VertexAIEmbeddings

from langchain_bigquery_hybridsearch import (
    BigQueryHybridSearchRetriever,
    BigQueryHybridSearchVectorStore,
)

### Load environment variables

We reuse the `.env` template from the integration tests
([tests/.env.example](../../tests/.env.example)). `find_dotenv(usecwd=True)`
walks up from the current working directory, so the file is picked up
regardless of where the notebook is launched.

In [2]:
load_dotenv(find_dotenv(usecwd=True))

PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"]
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
BQ_LOCATION = os.environ.get("BIGQUERY_LOCATION", LOCATION)
DATASET = os.environ.get("BIGQUERY_DATASET", "test_hybridsearch")
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL_NAME", "gemini-embedding-001")

# Use a unique table name per notebook run so reruns do not clash.
TABLE = os.environ.get("BIGQUERY_TABLE") or f"hybrid_demo_{uuid.uuid4().hex[:8]}"
AUX_TABLE = f"{TABLE}_pattern"  # used in section 7 (analyzer demo)

print(f"project        : {PROJECT_ID}")
print(f"bq location    : {BQ_LOCATION}")
print(f"dataset        : {DATASET}")
print(f"main table     : {TABLE}")
print(f"aux table      : {AUX_TABLE}")
print(f"embedding model: {EMBEDDING_MODEL}")

project        : *****
bq location    : us-central1
dataset        : test_hybridsearch
main table     : hybrid_demo_2ee02f0d
aux table      : hybrid_demo_2ee02f0d_pattern
embedding model: gemini-embedding-001


### Pretty-printer for results

In [3]:
def show(results, *, with_score: bool = False) -> None:
    """Print docs (and optionally scores) compactly."""
    if not results:
        print("(no results)")
        return
    for i, item in enumerate(results, start=1):
        if with_score:
            doc, score = item
            print(f"[{i}] score={score:.4f}  meta={doc.metadata}")
        else:
            doc = item
            print(f"[{i}] meta={doc.metadata}")
        print(f"    {doc.page_content}")

## 2. Create the vectorstore

`BigQueryHybridSearchVectorStore` extends `BigQueryVectorStore`, so the
BigQuery dataset and table are **auto-created on first use** — no manual
DDL required (see `langchain_google_community/bq_storage_vectorstores/_base.py`).
It also kicks off a background `CREATE SEARCH INDEX` if one does not exist;
the index can take a minute or two to become queryable.

In [4]:
embedding = VertexAIEmbeddings(
    model_name=EMBEDDING_MODEL,
    project=PROJECT_ID,
    location=LOCATION,
)

store = BigQueryHybridSearchVectorStore(
    project_id=PROJECT_ID,
    dataset_name=DATASET,
    table_name=TABLE,
    location=BQ_LOCATION,
    embedding=embedding,
    distance_type="COSINE",
    # All hybrid-specific defaults shown explicitly:
    search_analyzer="LOG_ANALYZER",
    hybrid_search_mode="pre_filter",
    rrf_k=60,
)
store

/tmp/ipykernel_4334/299674034.py:1: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embedding = VertexAIEmbeddings(
/tmp/ipykernel_4334/299674034.py:1: LangChainDeprecationWarning: The class `VertexAIEmbeddings` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import GoogleGenerativeAIEmbeddings``.
  embedding = VertexAIEmbeddings(


BigQuery table test_hybridsearch.hybrid_demo_2ee02f0d initialized/validated as persistent storage. Access via BigQuery console:
 https://console.cloud.google.com/bigquery?project=&ws=!1m5!1m4!4m3!1stest_hybridsearch!3shybrid_demo_2ee02f0d


BigQueryHybridSearchVectorStore(embedding=VertexAIEmbeddings(client=<google.genai.client.Client object at 0x7d8c1d187230>, project='*****', location='us-central1', model_name='gemini-embedding-001', credentials=None, max_retries=6, dimensions=None), project_id='*****', dataset_name='test_hybridsearch', table_name='hybrid_demo_2ee02f0d', location='us-central1', content_field='content', embedding_field='embedding', doc_id_field='doc_id', temp_dataset_name='test_hybridsearch_temp', credentials=None, embedding_dimension=3072, extra_fields=None, table_schema=None, distance_type='COSINE', search_fields=['content'], search_analyzer='LOG_ANALYZER', search_analyzer_options=None, hybrid_search_mode='pre_filter', rrf_k=60)

### Load sample documents

12 short passages spanning several Google Cloud products. Each has
`topic` and `year` metadata, which we use for filtering later.

In [5]:
texts = [
    "BigQuery is a serverless, highly scalable data warehouse by Google Cloud.",
    "BigQuery supports SQL queries over petabyte-scale datasets without infra.",
    "VECTOR_SEARCH finds semantically similar embeddings stored in BigQuery.",
    "The SEARCH function performs full-text keyword matching on indexed columns.",
    "Hybrid search combines keyword and semantic search for higher recall.",
    "Reciprocal Rank Fusion (RRF) merges ranked lists from independent retrievers.",
    "Cloud Spanner is a globally distributed, strongly consistent relational DB.",
    "Spanner offers horizontal scaling with external consistency guarantees.",
    "Vertex AI provides managed embedding models such as gemini-embedding-001.",
    "The text-embedding-005 model produces high-quality semantic vectors.",
    "AlloyDB for PostgreSQL accelerates analytical queries with columnar caches.",
    "Cloud Storage buckets can host static assets and act as a data lake source.",
]
metadatas = [
    {"source": "docs", "topic": "bigquery", "year": 2024},
    {"source": "docs", "topic": "bigquery", "year": 2025},
    {"source": "docs", "topic": "vector",   "year": 2025},
    {"source": "docs", "topic": "search",   "year": 2024},
    {"source": "docs", "topic": "hybrid",   "year": 2025},
    {"source": "docs", "topic": "hybrid",   "year": 2025},
    {"source": "docs", "topic": "spanner",  "year": 2024},
    {"source": "docs", "topic": "spanner",  "year": 2025},
    {"source": "docs", "topic": "vertex",   "year": 2025},
    {"source": "docs", "topic": "vertex",   "year": 2024},
    {"source": "docs", "topic": "alloydb",  "year": 2024},
    {"source": "docs", "topic": "storage",  "year": 2025},
]

ids = store.add_texts(texts=texts, metadatas=metadatas)
print(f"inserted {len(ids)} rows")

inserted 12 rows


## 3. Inherited similarity search (sanity check)

Before exercising the hybrid path, confirm the underlying
`BigQueryVectorStore` semantic search works on this table.

> **Note on scores** — Because `distance_type="COSINE"`, the returned `score` is a cosine **distance** (`1 − cosine_similarity`),
> so **smaller values mean greater similarity**. The same convention applies to `hybrid_search_with_score` in §4 (pre-filter mode).
> RRF mode in §5 is the opposite: **larger scores mean stronger relevance**. See [Score Semantics](../../README.md#score-semantics) in the README for details.

In [6]:
show(store.similarity_search("serverless data warehouse", k=3))

[1] meta={'doc_id': '0a7e41d8dc624d67b1e5deae97d637bf', 'source': 'docs', 'topic': 'bigquery', 'year': 2024, 'score': 0.233224700318159}
    BigQuery is a serverless, highly scalable data warehouse by Google Cloud.
[2] meta={'doc_id': '2cf3f606429c4658b7bdaadba7404c41', 'source': 'docs', 'topic': 'bigquery', 'year': 2025, 'score': 0.28062317348596}
    BigQuery supports SQL queries over petabyte-scale datasets without infra.
[3] meta={'doc_id': '834fd19a95524007b37bd468e7a38369', 'source': 'docs', 'topic': 'alloydb', 'year': 2024, 'score': 0.3143715240806645}
    AlloyDB for PostgreSQL accelerates analytical queries with columnar caches.


In [7]:
show(
    store.similarity_search_with_score("managed embedding model", k=3),
    with_score=True,
)

[1] score=0.2197  meta={'doc_id': '1528e0945fd34114958625d3d4a16b2e', 'source': 'docs', 'topic': 'vertex', 'year': 2025, 'score': 0.219693810683931}
    Vertex AI provides managed embedding models such as gemini-embedding-001.
[2] score=0.2959  meta={'doc_id': '8fba0e3a4ad24afbb32bb6b9dc9c0af1', 'source': 'docs', 'topic': 'vertex', 'year': 2024, 'score': 0.29592968290459776}
    The text-embedding-005 model produces high-quality semantic vectors.
[3] score=0.3334  meta={'doc_id': 'ccd0cae7e86f4a4c8e03b8ec3776af62', 'source': 'docs', 'topic': 'vector', 'year': 2025, 'score': 0.33337157797354067}
    VECTOR_SEARCH finds semantically similar embeddings stored in BigQuery.


## 4. Hybrid search — pre-filter mode

```
Query → SEARCH(content, keywords) → filtered rows → VECTOR_SEARCH() → top-k
```

Results **must** contain the keyword tokens; among those, vector distance
decides the order. The reported score is the vector distance.

In [8]:
# Default mode is already "pre_filter"; making it explicit for clarity.
show(store.hybrid_search(
    query="How does BigQuery serverless work?",
    text_query="BigQuery",
    k=3,
    hybrid_search_mode="pre_filter",
))

[1] meta={'doc_id': '0a7e41d8dc624d67b1e5deae97d637bf', 'source': 'docs', 'topic': 'bigquery', 'year': 2024}
    BigQuery is a serverless, highly scalable data warehouse by Google Cloud.
[2] meta={'doc_id': '2cf3f606429c4658b7bdaadba7404c41', 'source': 'docs', 'topic': 'bigquery', 'year': 2025}
    BigQuery supports SQL queries over petabyte-scale datasets without infra.
[3] meta={'doc_id': 'ccd0cae7e86f4a4c8e03b8ec3776af62', 'source': 'docs', 'topic': 'vector', 'year': 2025}
    VECTOR_SEARCH finds semantically similar embeddings stored in BigQuery.


### Same call, scores included

In [9]:
show(
    store.hybrid_search_with_score(
        query="How does BigQuery serverless work?",
        text_query="BigQuery",
        k=3,
    ),
    with_score=True,
)

[1] score=0.2317  meta={'doc_id': '0a7e41d8dc624d67b1e5deae97d637bf', 'source': 'docs', 'topic': 'bigquery', 'year': 2024}
    BigQuery is a serverless, highly scalable data warehouse by Google Cloud.
[2] score=0.2697  meta={'doc_id': '2cf3f606429c4658b7bdaadba7404c41', 'source': 'docs', 'topic': 'bigquery', 'year': 2025}
    BigQuery supports SQL queries over petabyte-scale datasets without infra.
[3] score=0.3556  meta={'doc_id': 'ccd0cae7e86f4a4c8e03b8ec3776af62', 'source': 'docs', 'topic': 'vector', 'year': 2025}
    VECTOR_SEARCH finds semantically similar embeddings stored in BigQuery.


### Decoupling the semantic query from the keyword query

`text_query` is what `SEARCH()` filters on; `query` is what gets embedded.
Below: rank by *vector* relevance to `serverless analytics` while *requiring*
the keyword `Spanner` — useful when you want results that mention a term but
are still ordered by a different intent.

In [10]:
show(
    store.hybrid_search_with_score(
        query="serverless analytics",
        text_query="Spanner",
        k=3,
    ),
    with_score=True,
)

[1] score=0.3722  meta={'doc_id': '61fd95f94b4e41a394d38764cffe9ead', 'source': 'docs', 'topic': 'spanner', 'year': 2024}
    Cloud Spanner is a globally distributed, strongly consistent relational DB.
[2] score=0.3798  meta={'doc_id': 'daba17f7ca774f9facf41c65cc8eba7f', 'source': 'docs', 'topic': 'spanner', 'year': 2025}
    Spanner offers horizontal scaling with external consistency guarantees.


### Empty result when keyword has no match

If `SEARCH()` filters everything out, `VECTOR_SEARCH()` has nothing to rank,
so pre-filter mode returns an empty list (no fallback).

In [11]:
show(store.hybrid_search(
    query="data warehouse",
    text_query="xyznonexistent123",
    k=3,
))

(no results)


## 5. Hybrid search — RRF mode

```
Query → VECTOR_SEARCH() → vector_rank ─┐
                                       ├→ rrf_score → top-k
Query → SEARCH()         → text_rank  ─┘
```

$$\text{rrf\_score} = \frac{1}{k + \text{vector\_rank}} + \frac{1}{k + \text{text\_rank}}$$

where the constant `k` is the instance's `rrf_k` (default `60`).
Unlike pre-filter, RRF returns rows that match **either** retriever, so it
tolerates keyword misses.

In [12]:
show(
    store.hybrid_search_with_score(
        query="semantic embedding similarity",
        text_query="VECTOR_SEARCH",
        k=5,
        fetch_k=25,
        hybrid_search_mode="rrf",
    ),
    with_score=True,
)

[1] score=0.0325  meta={'doc_id': 'ccd0cae7e86f4a4c8e03b8ec3776af62'}
    VECTOR_SEARCH finds semantically similar embeddings stored in BigQuery.
[2] score=0.0164  meta={'doc_id': '8fba0e3a4ad24afbb32bb6b9dc9c0af1'}
    The text-embedding-005 model produces high-quality semantic vectors.
[3] score=0.0159  meta={'doc_id': '1528e0945fd34114958625d3d4a16b2e'}
    Vertex AI provides managed embedding models such as gemini-embedding-001.
[4] score=0.0156  meta={'doc_id': 'dcd377e7ee6842e38d85e1783ec4929b'}
    Hybrid search combines keyword and semantic search for higher recall.
[5] score=0.0154  meta={'doc_id': 'e4f5cc7878ef483995e4d09cb7db24e4'}
    Reciprocal Rank Fusion (RRF) merges ranked lists from independent retrievers.


### `fetch_k` controls the candidate pool per retriever

Each leg (`VECTOR_SEARCH` and `SEARCH`) returns `fetch_k` rows; they are then
merged and the top `k` survive. A smaller `fetch_k` is cheaper but may miss
matches that only one retriever would surface.

In [13]:
for fetch_k in (5, 25):
    print(f"--- fetch_k = {fetch_k} ---")
    show(
        store.hybrid_search_with_score(
            query="distributed relational database",
            text_query="hybrid",
            k=4,
            fetch_k=fetch_k,
            hybrid_search_mode="rrf",
        ),
        with_score=True,
    )
    print()

--- fetch_k = 5 ---
[1] score=0.0164  meta={'doc_id': 'dcd377e7ee6842e38d85e1783ec4929b'}
    Hybrid search combines keyword and semantic search for higher recall.
[2] score=0.0164  meta={'doc_id': '61fd95f94b4e41a394d38764cffe9ead'}
    Cloud Spanner is a globally distributed, strongly consistent relational DB.
[3] score=0.0161  meta={'doc_id': 'daba17f7ca774f9facf41c65cc8eba7f'}
    Spanner offers horizontal scaling with external consistency guarantees.
[4] score=0.0159  meta={'doc_id': '2cf3f606429c4658b7bdaadba7404c41'}
    BigQuery supports SQL queries over petabyte-scale datasets without infra.

--- fetch_k = 25 ---
[1] score=0.0307  meta={'doc_id': 'dcd377e7ee6842e38d85e1783ec4929b'}
    Hybrid search combines keyword and semantic search for higher recall.
[2] score=0.0164  meta={'doc_id': '61fd95f94b4e41a394d38764cffe9ead'}
    Cloud Spanner is a globally distributed, strongly consistent relational DB.
[3] score=0.0161  meta={'doc_id': 'daba17f7ca774f9facf41c65cc8eba7f'}
    Sp

## 6. Tuning `rrf_k`

`rrf_k` lives on the *store* (not the call), so we instantiate a second
store pointing at the **same table**. With `rrf_k=10` the score gap between
rank 1 and rank 5 widens, so the highest-ranked retriever dominates.

In [14]:
store_low_k = BigQueryHybridSearchVectorStore(
    project_id=PROJECT_ID,
    dataset_name=DATASET,
    table_name=TABLE,
    location=BQ_LOCATION,
    embedding=embedding,
    distance_type="COSINE",
    rrf_k=10,
)

query, text_query = "semantic embedding similarity", "VECTOR_SEARCH"

print("=== rrf_k = 60 (default) ===")
show(
    store.hybrid_search_with_score(
        query=query, text_query=text_query, k=4, hybrid_search_mode="rrf",
    ),
    with_score=True,
)
print("\n=== rrf_k = 10 ===")
show(
    store_low_k.hybrid_search_with_score(
        query=query, text_query=text_query, k=4, hybrid_search_mode="rrf",
    ),
    with_score=True,
)

BigQuery table test_hybridsearch.hybrid_demo_2ee02f0d initialized/validated as persistent storage. Access via BigQuery console:
 https://console.cloud.google.com/bigquery?project=&ws=!1m5!1m4!4m3!1stest_hybridsearch!3shybrid_demo_2ee02f0d
=== rrf_k = 60 (default) ===
[1] score=0.0325  meta={'doc_id': 'ccd0cae7e86f4a4c8e03b8ec3776af62'}
    VECTOR_SEARCH finds semantically similar embeddings stored in BigQuery.
[2] score=0.0164  meta={'doc_id': '8fba0e3a4ad24afbb32bb6b9dc9c0af1'}
    The text-embedding-005 model produces high-quality semantic vectors.
[3] score=0.0159  meta={'doc_id': '1528e0945fd34114958625d3d4a16b2e'}
    Vertex AI provides managed embedding models such as gemini-embedding-001.
[4] score=0.0156  meta={'doc_id': 'dcd377e7ee6842e38d85e1783ec4929b'}
    Hybrid search combines keyword and semantic search for higher recall.

=== rrf_k = 10 ===
[1] score=0.1742  meta={'doc_id': 'ccd0cae7e86f4a4c8e03b8ec3776af62'}
    VECTOR_SEARCH finds semantically similar embeddings store

## 7. Metadata filtering

`filter` accepts either a dict (`AND` of equalities) or a raw SQL `WHERE`
expression. Works in both pre-filter and RRF modes.

### Dict filter — pre-filter mode

In [15]:
show(store.hybrid_search(
    query="vector store",
    text_query="VECTOR_SEARCH",
    k=5,
    filter={"topic": "vector"},
))

[1] meta={'doc_id': 'ccd0cae7e86f4a4c8e03b8ec3776af62', 'source': 'docs', 'topic': 'vector', 'year': 2025}
    VECTOR_SEARCH finds semantically similar embeddings stored in BigQuery.


### Raw SQL filter — RRF mode

In [16]:
show(store.hybrid_search(
    query="managed analytics",
    text_query="BigQuery",
    k=5,
    filter="year >= 2025 AND topic != 'spanner'",
    hybrid_search_mode="rrf",
))

[1] meta={'doc_id': '2cf3f606429c4658b7bdaadba7404c41'}
    BigQuery supports SQL queries over petabyte-scale datasets without infra.
[2] meta={'doc_id': 'ccd0cae7e86f4a4c8e03b8ec3776af62'}
    VECTOR_SEARCH finds semantically similar embeddings stored in BigQuery.
[3] meta={'doc_id': '1528e0945fd34114958625d3d4a16b2e'}
    Vertex AI provides managed embedding models such as gemini-embedding-001.
[4] meta={'doc_id': '901a08dcffe941168d7ca17b8542a8a3'}
    Cloud Storage buckets can host static assets and act as a data lake source.
[5] meta={'doc_id': 'dcd377e7ee6842e38d85e1783ec4929b'}
    Hybrid search combines keyword and semantic search for higher recall.


### Filter that excludes everything

In [17]:
show(store.hybrid_search(
    query="anything",
    text_query="BigQuery",
    k=5,
    filter={"topic": "nonexistent"},
))

(no results)


## 8. Analyzer and search-field configuration

BigQuery `SEARCH()` supports three text analyzers:

| analyzer | behaviour |
|---|---|
| `LOG_ANALYZER` (default) | tokenizes by whitespace + punctuation, lowercases |
| `NO_OP_ANALYZER` | exact-match only |
| `PATTERN_ANALYZER` | regex-driven tokenization (configure via `search_analyzer_options`) |

Below we create a *separate* table that uses `PATTERN_ANALYZER` so we can
compare against the `LOG_ANALYZER` defaults on the main table.

**Multi-column `SEARCH()`** — `search_fields` accepts more than one column
(e.g. `["title", "content"]`). The base store stores all metadata in a
single JSON-typed column by default, so multi-column hybrid search is most
useful when you provision additional typed columns via `extra_fields` on
`BigQueryVectorStore`. That setup is out of scope here; the call shape is:

```python
BigQueryHybridSearchVectorStore(
    ...,
    extra_fields={"title": "STRING"},
    search_fields=["title", "content"],
)
```

In [18]:
# A pattern that splits on whitespace and keeps alphanumeric runs only.
ANALYZER_OPTIONS = '{"patterns": ["[a-zA-Z0-9]+"]}'

store_pattern = BigQueryHybridSearchVectorStore(
    project_id=PROJECT_ID,
    dataset_name=DATASET,
    table_name=AUX_TABLE,
    location=BQ_LOCATION,
    embedding=embedding,
    distance_type="COSINE",
    search_analyzer="PATTERN_ANALYZER",
    search_analyzer_options=ANALYZER_OPTIONS,
)

store_pattern.add_texts(texts=texts, metadatas=metadatas)

show(store_pattern.hybrid_search(
    query="keyword search",
    text_query="SEARCH",
    k=3,
))

BigQuery table test_hybridsearch.hybrid_demo_2ee02f0d_pattern initialized/validated as persistent storage. Access via BigQuery console:
 https://console.cloud.google.com/bigquery?project=&ws=!1m5!1m4!4m3!1stest_hybridsearch!3shybrid_demo_2ee02f0d_pattern


[1] meta={'doc_id': '6550ec7aaaa34aafac72ced0d31e8b11', 'source': 'docs', 'topic': 'search', 'year': 2024}
    The SEARCH function performs full-text keyword matching on indexed columns.
[2] meta={'doc_id': 'e75cfd08910e4a5985ae73e5d51ddedc', 'source': 'docs', 'topic': 'search', 'year': 2024}
    The SEARCH function performs full-text keyword matching on indexed columns.
[3] meta={'doc_id': '3ea61d2cc80e4364a8605d6b08cf119c', 'source': 'docs', 'topic': 'search', 'year': 2024}
    The SEARCH function performs full-text keyword matching on indexed columns.


## 9. Retriever interface (LCEL-ready)

`store.as_retriever(...)` returns a `BigQueryHybridSearchRetriever`, which
extends `VectorStoreRetriever`. It accepts a new `search_type="hybrid"` in
addition to the standard `"similarity"`, `"mmr"`, and
`"similarity_score_threshold"`.

In [19]:
hybrid_retriever = store.as_retriever(
    search_type="hybrid",
    search_kwargs={
        "k": 4,
        "text_query": "BigQuery",
        "hybrid_search_mode": "pre_filter",
    },
)

isinstance(hybrid_retriever, BigQueryHybridSearchRetriever), hybrid_retriever.search_type

(True, 'hybrid')

In [20]:
docs = hybrid_retriever.invoke("How does serverless data warehouse work?")
show(docs)

[1] meta={'doc_id': '0a7e41d8dc624d67b1e5deae97d637bf', 'source': 'docs', 'topic': 'bigquery', 'year': 2024}
    BigQuery is a serverless, highly scalable data warehouse by Google Cloud.
[2] meta={'doc_id': '2cf3f606429c4658b7bdaadba7404c41', 'source': 'docs', 'topic': 'bigquery', 'year': 2025}
    BigQuery supports SQL queries over petabyte-scale datasets without infra.
[3] meta={'doc_id': 'ccd0cae7e86f4a4c8e03b8ec3776af62', 'source': 'docs', 'topic': 'vector', 'year': 2025}
    VECTOR_SEARCH finds semantically similar embeddings stored in BigQuery.


### Standard search types still work on the same store

In [21]:
sim_retriever = store.as_retriever(search_type="similarity", search_kwargs={"k": 3})
show(sim_retriever.invoke("managed embedding"))

print()
mmr_retriever = store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 8, "lambda_mult": 0.5},
)
show(mmr_retriever.invoke("managed embedding"))

[1] meta={'doc_id': '1528e0945fd34114958625d3d4a16b2e', 'source': 'docs', 'topic': 'vertex', 'year': 2025, 'score': 0.26285874524818764}
    Vertex AI provides managed embedding models such as gemini-embedding-001.
[2] meta={'doc_id': '8fba0e3a4ad24afbb32bb6b9dc9c0af1', 'source': 'docs', 'topic': 'vertex', 'year': 2024, 'score': 0.3380580453171407}
    The text-embedding-005 model produces high-quality semantic vectors.
[3] meta={'doc_id': 'ccd0cae7e86f4a4c8e03b8ec3776af62', 'source': 'docs', 'topic': 'vector', 'year': 2025, 'score': 0.35031169724515254}
    VECTOR_SEARCH finds semantically similar embeddings stored in BigQuery.

[1] meta={'doc_id': '1528e0945fd34114958625d3d4a16b2e', 'source': 'docs', 'topic': 'vertex', 'year': 2025, 'score': 0.26285874524818764}
    Vertex AI provides managed embedding models such as gemini-embedding-001.
[2] meta={'doc_id': 'daba17f7ca774f9facf41c65cc8eba7f', 'source': 'docs', 'topic': 'spanner', 'year': 2025, 'score': 0.3867337159558325}
    Spanne

### Use the retriever inside an LCEL chain

The retriever plugs into any LangChain chain. Example skeleton (not run here
to avoid pulling in an LLM dependency):

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_google_vertexai import ChatVertexAI

prompt = ChatPromptTemplate.from_template(
    "Answer using only the context.\n\nContext:\n{context}\n\nQ: {question}"
)
llm = ChatVertexAI(model_name="gemini-2.5-flash", project=PROJECT_ID, location=LOCATION)

chain = (
    {"context": hybrid_retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
chain.invoke("What is hybrid search?")
```

## 10. Cleanup

Drops the two tables created above. The dataset itself is left in place; to
remove it run `bq rm --dataset --force <project>:<dataset>` from a shell.

In [ ]:
client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
for tbl in (TABLE, AUX_TABLE):
    full = f"{PROJECT_ID}.{DATASET}.{tbl}"
    client.delete_table(full, not_found_ok=True)
    print(f"deleted {full}")